# Sentiment Labeling Pipeline

**Steps:**

1. Load `datasets/final/mda_shared_preprocessed.parquet`, explode `sentences`
2. Drop duplicate sentences, then create a randomized **400-sentence** manual sample with positive/negative/neutral coverage
3. Sample **50k sentences** → LLM few-shot labels (inter-rater check; reuses cached results when available)
4. Merge with your manual labels → Cohen's Kappa
   - κ > 0.7 → proceed; else revise prompt in Section 3
5. Label **entire dataset** with LLM
6. Check class proportions (>80% dominant → prompt-refinement placeholder)
7. Spot-check 100 random sentences → Excel
8. Save final labeled dataset

**Labeling guidelines:**

- Clear growth signal / positive factual outcome → **positive**
- Negative factual outcome → **negative**
- Forward-looking statements → **neutral**
- Backward-looking negative statements → **negative**
- Otherwise → **neutral**


## 0. Imports & Config


In [1]:
# Fast dependency install via uv into a repo-local package directory.
import shutil
import subprocess
import sys
from pathlib import Path

PACKAGE_DIR = Path.cwd().resolve() / ".uv_packages"
PACKAGE_DIR.mkdir(exist_ok=True)

if shutil.which("uv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "uv", "-q"])

subprocess.check_call(
    [
        "uv",
        "pip",
        "install",
        "--target",
        str(PACKAGE_DIR),
        "--no-cache",
        "--upgrade",
        "numpy==1.26.4",
        "pandas==2.2.3",
        "pyarrow>=15,<19",
        "openpyxl",
        "typing_extensions>=4.12.2",
        "pydantic>=2.7",
        "pydantic-core>=2.18",
        "tqdm",
        "vllm",
    ]
)

sys.path.insert(0, str(PACKAGE_DIR))
print(f"Using repo-local package directory: {PACKAGE_DIR}")
print("If this cell changed numpy/pandas, restart the kernel before running imports.")


Using CPython 3.11.11 interpreter at: jupyterlab-venv-pytorch-240/bin/python3


Using repo-local package directory: /common/home/users/w/wlcheng.2023/.uv_packages
If this cell changed numpy/pandas, restart the kernel before running imports.


Resolved 171 packages in 1.30s
Checked 171 packages in 3ms


In [2]:
import ast
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ── Paths ───────────────────────────────────────────────────────────────────────────────────
DATA_DIR = Path.home() / "sentiment_analysis_data"
DATA_PATH = DATA_DIR / "mda_shared_preprocessed.parquet"
MANUAL_OUT = DATA_DIR / "manual_labeling_sample.xlsx"
MANUAL_LABELED_PATH = DATA_DIR / "manual_labeling_400_labeled.xlsx"
FINAL_OUT = DATA_DIR / "labeled_dataset.csv"
SPOT_CHECK_OUT = DATA_DIR / "spot_check_100.xlsx"

# ── Constants ──────────────────────────────────────────────────────────────────────────────
VALID_LABELS = {"positive", "negative", "neutral"}
SEED = 42
NUM_GPUS = (
    len(os.environ["CUDA_VISIBLE_DEVICES"].split(","))
    if os.environ.get("CUDA_VISIBLE_DEVICES")
    else 1
)
CHUNK_SIZE = 512
random.seed(SEED)
np.random.seed(SEED)


def read_label_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)
    if "manual_label" in df.columns:
        df["manual_label"] = df["manual_label"].astype(str).str.strip()
    return df

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


## 1. Load Parquet & Explode Sentences


In [3]:
df_raw = pd.read_parquet(DATA_PATH)
print(f"✅ Loaded: {len(df_raw):,} filings")

df = (
    df_raw[
        [
            "doc_id",
            "company_name",
            "filing_type",
            "filing_date",
            "year",
            "quarter",
            "sentences",
        ]
    ]
    .explode("sentences")
    .rename(columns={"sentences": "sentence"})
    .dropna(subset=["sentence"])
    .reset_index(drop=True)
)
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"].str.len() > 20].reset_index(drop=True)
print(f"✅ Sentences: {len(df):,}")
df.head(3)


✅ Loaded: 18,169 filings
✅ Sentences: 1,251,937


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...


## 2. Deduplicate + Randomized 400-Sentence Sample (with Class Coverage)


In [4]:
dup_counts = df.duplicated(subset=["sentence"], keep=False).sum()
print(f"Total rows: {len(df):,} | Duplicate rows: {dup_counts:,}")

if dup_counts:
    top5 = (
        df[df.duplicated(subset=["sentence"], keep=False)]
        .groupby("sentence")
        .size()
        .nlargest(5)
    )
    for sent, cnt in top5.items():
        print(f"  {cnt}x | {sent[:120]}{'...' if len(sent) > 120 else ''}")


Total rows: 1,251,937 | Duplicate rows: 1,003,558
  1140x | actual results may differ from these estimates under different assumptions or conditions.
  1016x | we evaluate our estimates and assumptions on an ongoing basis.
  652x | our actual results may differ from these estimates under different assumptions or conditions.
  598x | we may be required to seek additional equity or debt financing.
  532x | on an ongoing basis we evaluate our estimates and assumptions.


In [5]:
MANUAL_N = 400

POS_HINTS = (
    "increased",
    "increase",
    "improved",
    "improvement",
    "growth",
    "grew",
    "strong",
    "record",
    "higher",
    "gain",
    "expanded",
    "expansion",
    "profit",
    "profitable",
    "beat",
    "outperformed",
)
NEG_HINTS = (
    "decreased",
    "decrease",
    "declined",
    "decline",
    "lower",
    "loss",
    "impairment",
    "charge",
    "charges",
    "restructuring",
    "adverse",
    "weak",
    "headwind",
    "downturn",
    "reduced",
    "fell",
    "drop",
)
NEUTRAL_HINTS = (
    "expect",
    "expects",
    "expected",
    "anticipate",
    "may",
    "might",
    "could",
    "will",
    "plan",
    "plans",
    "guidance",
    "outlook",
    "forecast",
)


def heuristic_bucket(s: str) -> str:
    s = s.lower()
    if any(w in s for w in NEUTRAL_HINTS):
        return "neutral"
    pos = any(w in s for w in POS_HINTS)
    neg = any(w in s for w in NEG_HINTS)
    if pos and not neg:
        return "positive"
    if neg and not pos:
        return "negative"
    return "neutral"


before = len(df)
df = df.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
print(f"✅ Removed {before - len(df):,} duplicates → {len(df):,} unique sentences")

if MANUAL_LABELED_PATH.exists():
    manual_sample = read_label_table(MANUAL_LABELED_PATH).copy()
    if "sample_bucket" not in manual_sample.columns:
        manual_sample["sample_bucket"] = manual_sample["sentence"].apply(
            heuristic_bucket
        )
    manual_sample["manual_label"] = (
        manual_sample["manual_label"].fillna("").astype(str).str.strip().str.lower()
    )
else:
    df["sample_bucket"] = df["sentence"].apply(heuristic_bucket)
    target = MANUAL_N // 3
    parts = [
        grp.sample(n=min(target, len(grp)), random_state=SEED)
        for lbl, grp in df.groupby("sample_bucket")
    ]
    manual_sample = pd.concat(parts, ignore_index=True)
    shortfall = MANUAL_N - len(manual_sample)
    if shortfall > 0:
        rest = df[~df.index.isin(manual_sample.index)]
        manual_sample = pd.concat(
            [
                manual_sample,
                rest.sample(n=min(shortfall, len(rest)), random_state=SEED),
            ],
            ignore_index=True,
        )
    manual_sample = manual_sample.sample(frac=1, random_state=SEED).reset_index(
        drop=True
    )
    manual_sample["manual_label"] = ""
    manual_sample.drop(columns=["sample_bucket"]).to_excel(MANUAL_OUT, index=False)
    print(f"✅ Manual sample written → {MANUAL_OUT}")
    print(manual_sample["sample_bucket"].value_counts().to_string())

manual_sample[
    ["sentence", "sample_bucket", "company_name", "year", "manual_label"]
].head(8)


✅ Removed 799,547 duplicates → 452,390 unique sentences


,sentence,sample_bucket,company_name,year,manual_label
0,net income was reduced in NUM due to NUM of af...,negative,SailPoint,2014,negative
1,jabil produces our printers to our design spec...,neutral,Zebra_Technologies,2011,neutral
2,the net deferred tax liability from the acquis...,positive,Salesforce,2013,positive
3,we buy xol so that the probability of losses f...,negative,Hippo,2025,neutral
4,the increase in interest income and other net ...,positive,KLA,2011,positive
5,these employees are automatically highlighted ...,positive,Genpact,2015,neutral
6,NUM acquisition-related costs represent fees a...,neutral,Sabre,2021,neutral
7,the increase was primarily driven by the trans...,positive,Lemonade,2024,neutral


## 3. LLM Label Manual Sample (Inter-rater Check)


In [ ]:
import re
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier for SEC 10-K/10-Q filing sentences.\n"
    "Return exactly one label: positive, negative, or neutral.\n\n"
    "Before answering, reason silently using this checklist:\n"
    "1. Is the sentence a clear realized event in the reported period?\n"
    "2. Is the net effect positive, negative, or unclear?\n"
    "3. Is the sentence instead forward-looking, definitional, strategic, mixed, or table-like?\n"
    "4. If both positive and negative signals appear and neither clearly dominates, choose neutral.\n\n"
    "Do not output the checklist or any explanation.\n"
    "Output only one word: positive, negative, or neutral.\n\n"
    "Use this decision order:\n"
    "1. Label neutral if the sentence is not a clear realized economic event in the reported period.\n"
    "   Neutral includes: forward-looking, hypothetical, conditional, definitional, note references, "
    "  accounting mechanics, table-like numeric fragments, and garbled fragments.\n"
    "2. Label neutral if the sentence contains both positive and negative signals and neither direction clearly dominates.\n"
    "3. Label negative if the sentence states a clear realized deterioration in financial performance, a realized increase "
    "   in harmful expenses or costs, a realized loss/charge/impairment/default, or a realized operational disruption.\n"
    "4. Label positive if the sentence states a clear realized improvement in financial performance or a realized decrease "
    "   in harmful expenses or costs.\n\n"
    "Respond with only one word: positive, negative, or neutral."
)

FEW_SHOT = (
    'Sentence: "Revenue increased from the prior-year period because customer demand strengthened." Label: positive\n'
    'Sentence: "Operating margin improved as lower operating costs more than offset higher labor expense." Label: positive\n'
    'Sentence: "Net income increased compared with the prior period." Label: positive\n'
    'Sentence: "Operating expenses declined because personnel and service costs were lower." Label: positive\n'
    'Sentence: "Operating income improved because sales growth more than offset higher operating costs." Label: positive\n'
    'Sentence: "Customer spending per account increased modestly from the prior quarter." Label: positive\n'
    'Sentence: "Gross profit declined as selling prices fell and input costs rose." Label: negative\n'
    'Sentence: "Revenue declined because customer demand weakened." Label: negative\n'
    'Sentence: "Earnings per share decreased from the prior-year quarter." Label: negative\n'
    'Sentence: "Order volume declined compared with the prior-year period." Label: negative\n'
    'Sentence: "The company recognized foreign exchange losses during the period." Label: negative\n'
    'Sentence: "Returns on earning assets declined from the comparable prior-year period." Label: negative\n'
    'Sentence: "Operations were disrupted by delivery delays and supply chain constraints during the quarter." Label: negative\n'
    'Sentence: "Interest expense increased due to higher debt balances and borrowing rates." Label: negative\n'
    'Sentence: "Non-cash expense related to acquired assets increased during the period." Label: negative\n'
    'Sentence: "Labor cost inflation increased relative to the prior-year period." Label: negative\n'
    'Sentence: "A bankruptcy filing triggered a default under a debt agreement." Label: negative\n'
    'Sentence: "Revenue per transaction declined because the sales mix shifted toward lower-value offerings." Label: negative\n'
    'Sentence: "Persistent financial pressure from weak sales and substantial debt obligations weighed on the business during the period." Label: negative\n'
    'Sentence: "Manufacturing output was limited by operational downtime during a production transition." Label: negative\n'
    'Sentence: "Operating performance was hurt by higher third-party service and travel costs." Label: negative\n'
    'Sentence: "Customer attrition in an indirect sales channel reduced period-over-period revenue." Label: negative\n'
    'Sentence: "Advertising and marketing expense increased to support business growth initiatives." Label: neutral\n'
    'Sentence: "Business-support spending increased as the company invested more in marketing and promotion." Label: neutral\n'
    'Sentence: "Changes in research and development spending related to project timing should be treated as neutral." Label: neutral\n'
    'Sentence: "Management expects to invest additional resources in the business over the next year." Label: neutral\n'
    'Sentence: "Prepaid expenses increased because the company made advance payments for future services." Label: neutral\n'
    'Sentence: "The pace of declines moderated during the quarter." Label: neutral\n'
    'Sentence: "See Note NUM for additional discussion." Label: neutral\n'
    'Sentence: "Revenue increased while expenses also increased, and the net effect was unclear." Label: neutral\n'
    'Sentence: "A decline that is partly explained by an offsetting business expansion benefit should be labeled neutral when the net effect is unclear." Label: neutral\n'
    'Sentence: "Strategic portfolio repositioning, without a clearly stated net financial impact, is neutral." Label: neutral\n'
    'Sentence: "Inventory levels adjusted to match near-term demand expectations should be labeled neutral." Label: neutral\n'
    'Sentence: "Risk-factor language about possible hiring, retention, or productivity challenges is neutral unless a realized impact is stated." Label: neutral\n'
    'Sentence: "Growth strategy, acquisitions, talent development, or expansion plans are neutral unless the sentence clearly states a realized improvement in financial performance." Label: neutral\n'
    'Sentence: "A table-like comparison line with many NUM values and little narrative explanation should be labeled neutral." Label: neutral\n'
)

FEW_SHOT = FEW_SHOT

# -----------------------------
# 2) MODEL LOAD
# -----------------------------
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"

if globals().get("_LOADED_MODEL_ID") != MODEL_ID:
    if "llm" in globals():
        del llm
    if "tokenizer" in globals():
        del tokenizer

    llm = LLM(
        model=MODEL_ID,
        quantization="awq_marlin",
        max_model_len=4096,
        tensor_parallel_size=NUM_GPUS,
        gpu_memory_utilization=0.95,
        enable_prefix_caching=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    _LOADED_MODEL_ID = MODEL_ID

# Constrained decoding: restrict output to only the first token of each valid label.
# The model generates immediately after "Label:" so the next token is
# " positive" / " negative" / " neutral". Limiting the vocabulary makes
# each decode step near-instant.
_label_token_ids = list(
    {
        ids[0]
        for word in [
            " positive",
            " negative",
            " neutral",
            "positive",
            "negative",
            "neutral",
        ]
        for ids in [tokenizer.encode(word, add_special_tokens=False)]
        if ids
    }
)
print(
    f"Constrained decoding: {len(_label_token_ids)} allowed token IDs -> {_label_token_ids}"
)

sampling_params = SamplingParams(
    temperature=0,
    max_tokens=1,
    allowed_token_ids=_label_token_ids,
)

VALID_LABELS = {"positive", "negative", "neutral"}

# -----------------------------
# 3) RULE-BASED GUARDRAILS
# -----------------------------
REF_PATTERNS = [
    r"\bsee\b.*\b(note|notes|financial statements?)\b",
    r"\brefer to\b.*\b(note|notes|financial statements?)\b",
    r"\bfor more details\b",
    r"\bcomputed by dividing\b",
    r"\bweighted-average number of shares\b",
    r"\bsee accompanying\b",
]

FORWARD_LOOKING_PATTERNS = [
    r"\bmay\b",
    r"\bmight\b",
    r"\bcould\b",
    r"\bwould\b",
    r"\bshould\b",
    r"\bexpect(?:s|ed)?\b",
    r"\banticipat(?:e|es|ed|ing)\b",
    r"\bplan(?:s|ned|ning)?\b",
    r"\bintend(?:s|ed|ing)?\b",
    r"\bif\b",
    r"\bfuture\b",
]

DEFINITIONAL_PATTERNS = [
    r"\bconsists? of\b",
    r"\bcomprised of\b",
    r"\bcomputed as\b",
    r"\bcalculated as\b",
    r"\bdefined as\b",
    r"\brepresents\b",
]

INVESTMENT_NEUTRAL_PATTERNS = [
    r"\badvertising and marketing\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\badvertising and marketing\b",
    r"\bmarketing\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\bmarketing\b",
    r"\br&d\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\br&d\b",
    r"\bresearch and development\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\bresearch and development\b",
    r"\bcapex\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\bcapex\b",
    r"\bcapital expenditures\b.*\bincrease",
    r"\bincrease(?:d)?\b.*\bcapital expenditures\b",
]

NEGATIVE_ATTRIBUTION_PATTERNS = [
    r"\b(revenue|sales|demand|margin|margins|profit|income|cash flow|eps|earnings per share|bookings|enrollment|yield)\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
    r"\b(decrease|decreased|decline|declined|fell|lower|reduced)\b.*\b(revenue|sales|demand|margin|margins|profit|income|cash flow|eps|earnings per share|bookings|enrollment|yield)\b",
]

POSITIVE_ATTRIBUTION_PATTERNS = [
    r"\b(revenue|sales|demand|margin|margins|profit|income|cash flow|eps|earnings per share|bookings|enrollment|yield)\b.*\b(increase|increased|improved|improvement|grew|growth|higher|expanded|expansion)\b",
    r"\b(increase|increased|improved|improvement|grew|growth|higher|expanded|expansion)\b.*\b(revenue|sales|demand|margin|margins|profit|income|cash flow|eps|earnings per share|bookings|enrollment|yield)\b",
]

MODERATION_PATTERNS = [
    r"\bdeclines? (have )?(slowed|moderated|stabilized)\b",
    r"\bdecreases? (have )?(slowed|moderated|stabilized)\b",
    r"\bmoderation of the declines?\b",
    r"\bmoderation of declines?\b",
    r"\bdeclines? moderated\b",
]

_DIRECTION_WORDS = frozenset(
    [
        "decrease",
        "decreased",
        "decline",
        "declined",
        "fell",
        "fallen",
        "drop",
        "dropped",
        "loss",
        "losses",
        "impairment",
        "charge",
        "charges",
        "write-off",
        "write-down",
        "writedown",
        "increase",
        "increased",
        "grew",
        "growth",
        "improved",
        "improvement",
        "gain",
        "higher",
        "lower",
        "reduced",
        "reduction",
        "delay",
        "delays",
        "disruption",
        "disruptions",
        "slippage",
        "shortage",
        "shortages",
    ]
)

FINANCIAL_SIGNAL_WORDS = frozenset(
    [
        "revenue",
        "sales",
        "demand",
        "margin",
        "margins",
        "profit",
        "income",
        "cash flow",
        "expense",
        "expenses",
        "cost",
        "costs",
        "sg&a",
        "interest expense",
        "operating expenses",
        "gross profit",
        "net income",
        "operating income",
        "eps",
        "earnings per share",
        "bookings",
        "enrollment",
        "yield",
        "loss",
        "losses",
    ]
)

HARD_NEGATIVE_PATTERNS = [
    r"\bforeign exchange transaction losses?\b",
    r"\bearnings per share\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
    r"\beps\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
    r"\bbookings\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
    r"\benrollment\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
    r"\byield\b.*\b(decrease|decreased|decline|declined|fell|lower|reduced)\b",
]

RND_NEUTRAL_PATTERNS = [
    r"\bresearch and development expense\b.*\b(increase|increased|decrease|decreased|decline|declined|lower|higher|reduced)\b",
    r"\br&d expense\b.*\b(increase|increased|decrease|decreased|decline|declined|lower|higher|reduced)\b",
    r"\b(increase|increased|decrease|decreased|decline|declined|lower|higher|reduced)\b.*\bresearch and development expense\b",
    r"\b(increase|increased|decrease|decreased|decline|declined|lower|higher|reduced)\b.*\br&d expense\b",
]

TABLE_LIKE_NUM_PATTERNS = [
    r"^for the .* ended .* revenues were num as compared to num .* decrease of num",
    r"^for the .* ended .* income was num as compared to num",
    r"^for the .* ended .* earnings per share .* as compared to",
    r"^num operating income num .* compared to num",
]

# Precompile regex to avoid repeated recompilation overhead during large-batch inference.
REF_REGEX = [re.compile(p) for p in REF_PATTERNS]
FORWARD_REGEX = [re.compile(p) for p in FORWARD_LOOKING_PATTERNS]
DEFINITIONAL_REGEX = [re.compile(p) for p in DEFINITIONAL_PATTERNS]
INVESTMENT_NEUTRAL_REGEX = [re.compile(p) for p in INVESTMENT_NEUTRAL_PATTERNS]
NEGATIVE_ATTRIBUTION_REGEX = [re.compile(p) for p in NEGATIVE_ATTRIBUTION_PATTERNS]
POSITIVE_ATTRIBUTION_REGEX = [re.compile(p) for p in POSITIVE_ATTRIBUTION_PATTERNS]
MODERATION_REGEX = [re.compile(p) for p in MODERATION_PATTERNS]
HARD_NEGATIVE_REGEX = [re.compile(p) for p in HARD_NEGATIVE_PATTERNS]
RND_NEUTRAL_REGEX = [re.compile(p) for p in RND_NEUTRAL_PATTERNS]
TABLE_LIKE_NUM_REGEX = [re.compile(p) for p in TABLE_LIKE_NUM_PATTERNS]


def _normalize(s: str) -> str:
    return " ".join(str(s).strip().lower().split())


def _any_match(compiled_patterns, s: str) -> bool:
    return any(p.search(s) for p in compiled_patterns)


def _looks_num_heavy_or_fragment(sentence: str) -> bool:
    toks = sentence.split()
    if not toks:
        return True

    num_count = sum(1 for t in toks if t.upper() == "NUM")
    ratio = num_count / len(toks)

    has_direction = any(w in sentence.lower() for w in _DIRECTION_WORDS)
    has_fin_signal = any(w in sentence.lower() for w in FINANCIAL_SIGNAL_WORDS)

    if ratio >= 0.20 or (num_count >= 4 and len(toks) <= 20):
        return not (has_direction and has_fin_signal)

    return False


def _is_reference_or_definition(s: str) -> bool:
    return _any_match(REF_REGEX, s) or _any_match(DEFINITIONAL_REGEX, s)


def _is_forward_looking(s: str) -> bool:
    return _any_match(FORWARD_REGEX, s)


def _is_investment_neutral(s: str) -> bool:
    if not _any_match(INVESTMENT_NEUTRAL_REGEX, s):
        return False
    return any(
        phrase in s
        for phrase in [
            "to support growth",
            "to support the business",
            "growth initiative",
            "growth initiatives",
            "expansion",
            "strategic",
            "invest",
            "investment",
        ]
    )


def _is_negative_attribution(s: str) -> bool:
    return _any_match(NEGATIVE_ATTRIBUTION_REGEX, s)


def _is_positive_attribution(s: str) -> bool:
    return _any_match(POSITIVE_ATTRIBUTION_REGEX, s)


def _is_moderation_neutral(s: str) -> bool:
    return _any_match(MODERATION_REGEX, s)


def _is_hard_negative(s: str) -> bool:
    return _any_match(HARD_NEGATIVE_REGEX, s)


def _is_rnd_neutral(s: str) -> bool:
    return _any_match(RND_NEUTRAL_REGEX, s)


def _is_table_like_num_line(s: str) -> bool:
    if _any_match(TABLE_LIKE_NUM_REGEX, s):
        return True

    toks = s.split()
    if not toks:
        return False

    num_count = sum(1 for t in toks if t == "num")
    ratio = num_count / len(toks)

    starts_like_period_line = s.startswith("for the ") or s.startswith("num ")
    has_as_compared = "as compared to" in s or "compared to" in s
    has_period = "ended" in s
    short_explanation = not any(
        phrase in s
        for phrase in [
            "due to",
            "primarily due to",
            "attributable to",
            "driven by",
            "because",
            "resulting from",
        ]
    )

    return (
        starts_like_period_line
        and has_as_compared
        and has_period
        and ratio >= 0.18
        and short_explanation
    )


def _policy_override(sentence: str, label: str) -> str:
    s = _normalize(sentence)

    # simple neutral filters
    if _is_reference_or_definition(s):
        return "neutral"

    if _is_forward_looking(s) and not any(
        x in s for x in ["resulted in", "have resulted in", "led to"]
    ):
        return "neutral"

    if _is_moderation_neutral(s):
        return "neutral"

    if _is_rnd_neutral(s):
        return "neutral"

    if _is_table_like_num_line(s):
        return "neutral"

    if _looks_num_heavy_or_fragment(s):
        return "neutral"

    # small number of high-precision overrides only
    if _is_investment_neutral(s):
        return "neutral"

    if _is_hard_negative(s):
        return "negative"

    if _is_negative_attribution(s):
        return "negative"

    return label


def _cheap_prelabel(sentence: str) -> str | None:
    s = _normalize(sentence)

    if _is_reference_or_definition(s):
        return "neutral"

    if _is_forward_looking(s) and not any(
        x in s for x in ["resulted in", "have resulted in", "led to"]
    ):
        return "neutral"

    if _is_moderation_neutral(s):
        return "neutral"

    if _is_rnd_neutral(s):
        return "neutral"

    if _is_table_like_num_line(s):
        return "neutral"

    if _looks_num_heavy_or_fragment(s):
        return "neutral"

    if _is_investment_neutral(s):
        return "neutral"

    if _is_hard_negative(s) or _is_negative_attribution(s):
        return "negative"

    if _is_positive_attribution(s) and not _is_negative_attribution(s):
        return "positive"

    return None


# -----------------------------
# 4) PROMPT BUILDER
# -----------------------------
def _build_prompt(sentence: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": FEW_SHOT + f'Sentence: "{sentence}" Label:'},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


# -----------------------------
# 5) LABELING
# -----------------------------
def _canonicalize_label(text: str) -> str:
    text = text.strip().lower()
    return text if text in VALID_LABELS else "neutral"


def _run_llm_pass(prompts: list[str], desc: str) -> list[str]:
    """Send all prompts to vLLM in one shot (no Python chunking loop).

    vLLM's internal scheduler does continuous batching far more efficiently
    than Python-level chunking. Constrained decoding (allowed_token_ids) makes
    each decode step near-instant because the softmax is over <=6 token IDs.
    """
    print(f"  Sending {len(prompts):,} prompts to LLM ({desc})...")
    results = llm.generate(prompts, sampling_params)
    return [out.outputs[0].text for out in results]


def llm_label_batch(
    sentences: list[str],
    use_stability_pass: bool = False,
    fast_mode: bool = True,
) -> list[str]:
    labels = [None] * len(sentences)

    unresolved_idx = []
    unresolved_sentences = []

    if fast_mode:
        for i, s in enumerate(sentences):
            quick = _cheap_prelabel(s)
            if quick is None:
                unresolved_idx.append(i)
                unresolved_sentences.append(s)
            else:
                labels[i] = quick
    else:
        unresolved_idx = list(range(len(sentences)))
        unresolved_sentences = sentences

    if not unresolved_sentences:
        return [x if x is not None else "neutral" for x in labels]

    if fast_mode:
        skipped = len(sentences) - len(unresolved_sentences)
        print(
            f"⚡ Fast-path pre-labeled {skipped:,}/{len(sentences):,} "
            f"({skipped / max(len(sentences), 1):.1%}); LLM on {len(unresolved_sentences):,}"
        )

    prompts_1 = [_build_prompt(s) for s in unresolved_sentences]
    raw_texts_1 = _run_llm_pass(prompts_1, "pass 1")

    labels_1 = [
        _policy_override(s, _canonicalize_label(t))
        for s, t in zip(unresolved_sentences, raw_texts_1)
    ]

    if not use_stability_pass:
        for idx, lbl in zip(unresolved_idx, labels_1):
            labels[idx] = lbl
        print(
            f"  Done: {pd.Series([x for x in labels if x is not None]).value_counts().to_dict()}"
        )
        return [x if x is not None else "neutral" for x in labels]

    raw_texts_2 = _run_llm_pass(prompts_1, "pass 2")

    labels_2 = [
        _policy_override(s, _canonicalize_label(t))
        for s, t in zip(unresolved_sentences, raw_texts_2)
    ]

    final_labels = [l1 if l1 == l2 else "neutral" for l1, l2 in zip(labels_1, labels_2)]

    for idx, lbl in zip(unresolved_idx, final_labels):
        labels[idx] = lbl

    print("  Pass1:", pd.Series(labels_1).value_counts().to_dict())
    print("  Pass2:", pd.Series(labels_2).value_counts().to_dict())
    print("  Final:", pd.Series(final_labels).value_counts().to_dict())
    return [x if x is not None else "neutral" for x in labels]


# -----------------------------
# 6) RUN
# -----------------------------
print(f"✅ Loaded model: {_LOADED_MODEL_ID}")
print(f"✅ LLM ready on {NUM_GPUS} GPU(s)")
print(f"✅ Labeling {len(manual_sample):,} manual sentences...")

llm_manual = manual_sample[["sentence"]].copy()
llm_manual["llm_label"] = llm_label_batch(
    manual_sample["sentence"].tolist(),
    use_stability_pass=True,
    fast_mode=False,
)

print(llm_manual["llm_label"].value_counts().to_string())

INFO 04-03 03:42:46 [utils.py:263] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
ERROR 04-03 03:43:00 [registry.py:700] Error saving model info cache.
ERROR 04-03 03:43:00 [registry.py:700] Traceback (most recent call last):
ERROR 04-03 03:43:00 [registry.py:700]   File "/common/home/users/w/wlcheng.2023/.uv_packages/vllm/model_executor/models/registry.py", line 692, in _save_modelinfo_to_cache
ERROR 04-03 03:43:00 [registry.py:700]     "modelinfo": asdict(mi),
ERROR 04-03 03:43:00 [registry.py:700]                  ^^^^^^^^^^
ERROR 04-03 03:43:00 [registry.py:700]   File "/opt/apps/software/Python/3.11.11-GCCcore-13.3.0/lib/python3.11/dataclasses.py", line 1286, in asdict
ERROR 04-03 03:43:00 [registry.py:700]     return _asdict_inner(obj, dict_factory)
ERROR 04-03 03:43:00 [registry.py:700]            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 04-03 03:43:01 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 04-03 03:43:01 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:03 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=408400) INFO 04-03 03:43:10 [default_loader.py:291] Loading weights took 3.84 seconds
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:11 [gpu_model_runner.py:3905] Model loading took 5.2 GiB memory and 5.859733 seconds
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:18 [backends.py:644] Using cache directory: /common/home/users/w/wlcheng.2023/.cache/vllm/torch_compile_cache/6d3fd1824a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:18 [backends.py:704] Dynamo bytecode transform time: 7.08 s
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:24 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.624 s
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:24 [monitor.py:34] torch.compile takes 8.70 s in total
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:25 [gpu_worker.py:358] Available KV cache memory: 15.7 GiB
(EngineCore_DP0 pid=408400) INFO 04-03 03:43:25 [kv_cache_utils.py:1305] GPU KV cache s

...

Processed prompts:   0%|          | 0/400 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Sending 400 prompts to LLM (pass 2)...


Adding requests:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/400 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Pass1: {'neutral': 283, 'negative': 70, 'positive': 47}
  Pass2: {'neutral': 283, 'negative': 70, 'positive': 47}
  Final: {'neutral': 283, 'negative': 70, 'positive': 47}
llm_label
neutral     283
negative     70
positive     47


## 4. Merge Manual Labels & Compute Cohen's Kappa

> **Action required before running this cell:**  
> Open `manual_labeling_sample.xlsx`, fill the `manual_label` column  
> (values: `positive` / `negative` / `neutral`), save as  
> `manual_labeling_sample_LABELED_COMPLETE.xlsx` in the same folder.


In [7]:
manual_labeled = read_label_table(MANUAL_LABELED_PATH)
manual_labeled["manual_label"] = (
    manual_labeled["manual_label"].fillna("").astype(str).str.strip().str.lower()
)
manual_labeled = manual_labeled[
    manual_labeled["manual_label"].isin(VALID_LABELS)
].copy()

merged = manual_labeled.merge(
    llm_manual[["sentence", "llm_label"]], on="sentence", how="inner"
)
print(f"✅ Matched for kappa: {len(merged):,} rows")

kappa = cohen_kappa_score(merged["manual_label"], merged["llm_label"])
print(f"\n✅ Cohen's Kappa = {kappa:.4f}")
print(
    "✅ Good agreement — proceed."
    if kappa > 0.7
    else "⚠️  Poor agreement — revise prompt in Section 3 and re-run."
)

pd.crosstab(
    merged["manual_label"],
    merged["llm_label"],
    rownames=["manual"],
    colnames=["llm"],
    margins=True,
)


✅ Matched for kappa: 400 rows

✅ Cohen's Kappa = 0.7831
✅ Good agreement — proceed.


llm,negative,neutral,positive,All
manual,,,,
negative,52,7,0,59
neutral,11,265,4,280
positive,7,11,43,61
All,70,283,47,400


In [8]:
disagreed = merged[merged["manual_label"] != merged["llm_label"]].copy()
print(f"❌ Disagreed rows: {len(disagreed):,} / {len(merged):,}")

cols = [
    c
    for c in [
        "sentence",
        "manual_label",
        "llm_label",
        "company_name",
        "year",
        "quarter",
        "filing_date",
    ]
    if c in disagreed.columns
]
disagreed = (
    disagreed[cols].sort_values(["manual_label", "llm_label"]).reset_index(drop=True)
)

pair_counts = (
    disagreed.groupby(["manual_label", "llm_label"]).size().sort_values(ascending=False)
)
print("\nTop disagreement pairs (manual -> llm):")
print(pair_counts.head(10).to_string())

# Save for prompt revision workflow.
out_path = DATA_DIR / "manual_vs_llm_disagreements.csv"
disagreed.to_csv(out_path, index=False)
print(f"\n✅ Saved disagreement table → {out_path}")

print("\nPreview (first 20 disagreements):")
preview_cols = [
    c for c in ["manual_label", "llm_label", "sentence"] if c in disagreed.columns
]
print(disagreed[preview_cols].head(20).to_string(index=False, max_colwidth=140))

disagreed.head(30)

❌ Disagreed rows: 40 / 400

Top disagreement pairs (manual -> llm):
manual_label  llm_label
neutral       negative     11
positive      neutral      11
negative      neutral       7
positive      negative      7
neutral       positive      4

✅ Saved disagreement table → /common/home/users/w/wlcheng.2023/sentiment_analysis_data/manual_vs_llm_disagreements.csv

Preview (first 20 disagreements):
manual_label llm_label                                                                                                                                     sentence
    negative   neutral the decrease in NUM as compared to NUM was primarily due to losses from two of our large mvnos during NUM approximately NUM of our wholes...
    negative   neutral amortization expense increased for the three months ended march NUM NUM as compared to the same period in NUM due to intangible assets ac...
    negative   neutral nursing wage inflation increased NUM while non-nursing wage inflation increased NUM in t

,sentence,manual_label,llm_label,company_name,year,quarter,filing_date
0,the decrease in NUM as compared to NUM was pri...,negative,neutral,SentinelOne,2011,Q1,2011-02-24
1,amortization expense increased for the three m...,negative,neutral,TechTarget,2022,Q2,2022-05-10
2,nursing wage inflation increased NUM while non...,negative,neutral,Gen_Digital,2019,Q2,2019-05-10
3,NUM macroeconomic factors we are aware that ne...,negative,neutral,NextNav,2023,Q1,2023-03-30
4,for several years we have experienced declinin...,negative,neutral,Lumen_Technologies,2025,Q1,2025-02-20
5,research and development expense NUM -weeks en...,negative,neutral,Garmin,2011,Q3,2011-08-03
6,other expense for the three months ended june ...,negative,neutral,Strategy____MicroStrategy_,2010,Q3,2010-08-04
7,these were followed in november NUM by an sec ...,neutral,negative,Strategy____MicroStrategy_,2024,Q1,2024-02-15
8,the increases for both the quarter and year-to...,neutral,negative,Teladoc_Health,2022,Q4,2022-11-02
9,we continue to rationalize our product portfol...,neutral,negative,ON_Semiconductor,2023,Q3,2023-07-31


In [9]:
print(f"Disagreed rows: {len(disagreed):,} / {len(merged):,}")
print(
    disagreed.groupby(["manual_label", "llm_label"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)


Disagreed rows: 40 / 400
manual_label  llm_label
neutral       negative     11
positive      neutral      11
negative      neutral       7
positive      negative      7
neutral       positive      4


## 5. Label Entire Dataset

Runs only if kappa > 0.7.


In [10]:
assert kappa > 0.7, f"Kappa = {kappa:.4f} ≤ 0.7. Revise prompt before proceeding."

df_full = df.copy()
print(f"✅ Labeling {len(df_full):,} sentences (high-speed inference mode)...")
df_full["llm_label"] = llm_label_batch(
    df_full["sentence"].tolist(),
    use_stability_pass=False,
    fast_mode=True,
)
print("\n✅ Label distribution:")
print(df_full["llm_label"].value_counts().to_string())

✅ Labeling 452,390 sentences (high-speed inference mode)...
⚡ Fast-path pre-labeled 179,938/452,390 (39.8%); LLM on 272,452
  Sending 272,452 prompts to LLM (pass 1)...


Adding requests:   0%|          | 0/272452 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/272452 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks…

  Done: {'neutral': 378386, 'positive': 37564, 'negative': 36440}

✅ Label distribution:
llm_label
neutral     378386
positive     37564
negative     36440


## 6. Check Class Proportions


In [11]:
props = df_full["llm_label"].value_counts(normalize=True)
print("✅ Class proportions:")
print(props.map("{:.2%}".format).to_string())

dom_cls = props.idxmax()
dom_pct = props.max()

if dom_pct > 0.80:
    print(f"\n⚠️  '{dom_cls}' dominates at {dom_pct:.1%}. Consider:")
    print("  1. Add more contrastive few-shot examples for minority classes")
    print("  2. Tighten/loosen the dominant-class definition in the prompt")
    print("  3. Re-run Section 3 with the revised prompt and re-check kappa")
else:
    print(f"\n✅ Class distribution looks healthy (no class > 80%).")


✅ Class proportions:
llm_label
neutral     83.64%
positive     8.30%
negative     8.05%

⚠️  'neutral' dominates at 83.6%. Consider:
  1. Add more contrastive few-shot examples for minority classes
  2. Tighten/loosen the dominant-class definition in the prompt
  3. Re-run Section 3 with the revised prompt and re-check kappa


## 8. Save Final Labeled Dataset


In [13]:
df_final = df_full[
    [
        "doc_id",
        "company_name",
        "filing_type",
        "filing_date",
        "year",
        "quarter",
        "sentence",
        "llm_label",
    ]
].rename(columns={"llm_label": "sentiment"})
df_final.to_csv(FINAL_OUT, index=False)

print(f"✅ Final dataset saved -> {FINAL_OUT}")
print(f"✅ Shape: {df_final.shape}")
print("\n✅ Label distribution:")
print(df_final["sentiment"].value_counts().to_string())
df_final.head(5)

✅ Final dataset saved -> /common/home/users/w/wlcheng.2023/sentiment_analysis_data/labeled_dataset.csv
✅ Shape: (452390, 8)

✅ Label distribution:
sentiment
neutral     378386
positive     37564
negative     36440


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral
